# Embedded AI Final Project

#### 20223265 조해영, 20214904 조성

0. 환경 설정

In [1]:
import json
import platform
import sys

import torch
import torchvision

print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print("GPU:", gpu_name)
else:
    gpu_name = None
    print("[warn] No GPU detected. Check Runtime > Change runtime type > GPU.")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ENV_INFO = {
    "python_version": sys.version.split()[0],
    "torch_version": torch.__version__,
    "torchvision_version": torchvision.__version__,
    "device": DEVICE,
    "gpu_name": gpu_name,
    "platform": platform.platform(),
}
print()
print(json.dumps(ENV_INFO, indent=2, ensure_ascii=False))

python: 3.13.9
torch: 2.14.0+cpu
torchvision: 0.29.0+cpu
cuda available: False
[warn] No GPU detected. Check Runtime > Change runtime type > GPU.

{
  "python_version": "3.13.9",
  "torch_version": "2.14.0+cpu",
  "torchvision_version": "0.29.0+cpu",
  "device": "cpu",
  "gpu_name": null,
  "platform": "Windows-11-10.0.26200-SP0"
}


### 1. 모델 호출

Classification을 위한 ViT를 호출한다

In [6]:
import timm
from timm.data import create_transform, resolve_model_data_config

MODEL_NAME = "vit_base_patch16_224"
model = timm.create_model(
    MODEL_NAME, 
    num_classes=1000,
    pretrained=True).to(DEVICE)
model.eval()

data_config = resolve_model_data_config(model)
eval_transform = create_transform(**data_config, is_training=False)

print("model:", MODEL_NAME)
print("device:", DEVICE)
print("preprocessing:", data_config)

model: vit_base_patch16_224
device: cpu
preprocessing: {'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.5, 0.5, 0.5), 'std': (0.5, 0.5, 0.5), 'crop_pct': 0.9, 'crop_mode': 'center'}


In [16]:
total_params = sum(parameter.numel() for parameter in model.parameters())
trainable_params = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
state_dict_size_bytes = sum(
    tensor.numel() * tensor.element_size() for tensor in model.state_dict().values()
)
state_dict_size_mib = state_dict_size_bytes / (1024 ** 2)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"State dict tensor size: {state_dict_size_mib:.2f} MiB ({state_dict_size_bytes:,} bytes)")

Total parameters: 86,567,656
Trainable parameters: 86,567,656
State dict tensor size: 330.23 MiB (346,270,624 bytes)


### ImageNet calibration / test 데이터

In [18]:
from pathlib import Path

from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

IMAGENETV2_ROOT = Path("./ImageNetV2-matched-frequency")
EXPECTED_CLASS_NAMES = {str(class_id) for class_id in range(1000)}
local_class_names = {
    entry.name for entry in IMAGENETV2_ROOT.iterdir() if entry.is_dir()
} if IMAGENETV2_ROOT.is_dir() else set()

if local_class_names == EXPECTED_CLASS_NAMES:
    class NumericClassImageFolder(ImageFolder):
        def find_classes(self, directory):
            classes = sorted(
                (entry.name for entry in Path(directory).iterdir() if entry.is_dir()),
                key=int,
            )
            return classes, {class_name: int(class_name) for class_name in classes}

    dataset = NumericClassImageFolder(IMAGENETV2_ROOT, transform=eval_transform)
    print(f"Using existing ImageNetV2 files: {IMAGENETV2_ROOT.resolve()}")
else:
    from imagenetv2_pytorch import ImageNetV2Dataset

    print("Local ImageNetV2 files not found; downloading with imagenetv2_pytorch.")
    dataset = ImageNetV2Dataset("matched-frequency")

dataloader = DataLoader(dataset, batch_size=32, shuffle=False)
print(f"Images loaded: {len(dataset)}")

Using existing ImageNetV2 files: C:\Users\82105\3-2\임베디드 AI\Embedded-AI-Final-Project\ImageNetV2-matched-frequency
Images loaded: 10000


In [ ]:
from pathlib import Path

from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder

# Local ImageNetV2 layout: ImageNetV2-matched-frequency/<class_id>/*.jpeg
IMAGENETV2_ROOT = Path("./ImageNetV2-matched-frequency")
CALIBRATION_RATIO = 0.2
BATCH_SIZE = 32
NUM_WORKERS = 0  # Reliable in notebooks on Windows.
SEED = 42
NUM_CLASSES = 1000

if not IMAGENETV2_ROOT.is_dir():
    raise FileNotFoundError(f"ImageNetV2 folder was not found: {IMAGENETV2_ROOT.resolve()}")


class NumericClassImageFolder(ImageFolder):
    def find_classes(self, directory):
        classes = [entry.name for entry in Path(directory).iterdir() if entry.is_dir()]
        if not all(class_name.isdigit() for class_name in classes):
            raise ValueError("ImageNetV2 class folders must be numeric IDs from 0 to 999.")
        classes.sort(key=int)
        expected_classes = [str(class_id) for class_id in range(NUM_CLASSES)]
        if classes != expected_classes:
            raise ValueError(
                f"Expected class folders 0-999, found {len(classes)} folders under {directory}."
            )
        return classes, {class_name: int(class_name) for class_name in classes}


dataset = NumericClassImageFolder(IMAGENETV2_ROOT, transform=eval_transform)
rng = torch.Generator().manual_seed(SEED)
class_indices = {class_id: [] for class_id in range(NUM_CLASSES)}
for index, class_id in enumerate(dataset.targets):
    class_indices[class_id].append(index)

calibration_indices = []
val_indices = []
calibration_counts = []
val_counts = []
for class_id, indices in class_indices.items():
    if len(indices) < 2:
        raise ValueError(f"Class {class_id} needs at least 2 images to split into both datasets.")
    shuffled = torch.randperm(len(indices), generator=rng).tolist()
    split_at = round(len(indices) * CALIBRATION_RATIO)
    split_at = min(max(split_at, 1), len(indices) - 1)
    calibration_indices.extend(indices[position] for position in shuffled[:split_at])
    val_indices.extend(indices[position] for position in shuffled[split_at:])
    calibration_counts.append(split_at)
    val_counts.append(len(indices) - split_at)

calibration_dataset = Subset(dataset, calibration_indices)
val_dataset = Subset(dataset, val_indices)

loader_options = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": DEVICE == "cuda",
}
calibration_loader = DataLoader(calibration_dataset, shuffle=False, **loader_options)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)

print(f"Dataset: {IMAGENETV2_ROOT.resolve()}")
print(f"ImageNet classes: {len(dataset.classes)}")
print(f"Calibration images: {len(calibration_dataset)}")
print(f"Test images: {len(val_dataset)}")
print(f"Per-class calibration counts: {sorted(set(calibration_counts))}")
print(f"Per-class test counts: {sorted(set(val_counts))}")
print(f"Image transform: {eval_transform}")

Dataset: C:\Users\82105\3-2\임베디드 AI\Embedded-AI-Final-Project\ImageNetV2-matched-frequency
ImageNet classes: 1000
Calibration images: 3000
Test images: 7000
Per-class calibration counts: [3]
Per-class test counts: [7]
Image transform: Compose(
    Resize(size=248, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    MaybeToTensor()
    Normalize(mean=tensor([0.5000, 0.5000, 0.5000]), std=tensor([0.5000, 0.5000, 0.5000]))
)


### 3. Baseline 모델 실행

In [15]:
model.eval()
correct = 0
total = 0

with torch.inference_mode():
    for images, labels in val_loader:
        images = images.to(DEVICE, non_blocking=DEVICE == "cuda")
        labels = labels.to(DEVICE, non_blocking=DEVICE == "cuda")

        predictions = model(images).argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

if total == 0:
    raise ValueError("Validation set is empty.")

val_accuracy = correct / total
print(f"Validation accuracy: {val_accuracy:.2%} ({correct}/{total})")

KeyboardInterrupt: 